# Lesson 04 - Tool Use Design Pattern

Sa leksiyon na ito matututuhan mo ang **Tool Use** design pattern para sa AI agents gamit ang Microsoft Agent Framework (Python). Tatalakayin natin:

- Pagtukoy ng function tools gamit ang `@tool` decorator at typed parameters
- Pagbibigay ng tool schemas para malaman ng model kung ano ang ginagawa ng bawat tool
- Pagkontrol ng tool execution gamit ang `approval_mode`
- Pagbabalik ng **structured output** sa pamamagitan ng mga Pydantic models at `response_format`

Ang senaryo ay isang **travel booking agent** na maaaring maghanap ng mga destinasyon, suriin ang availability, at kunin ang impormasyon ng mga flight.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pagde-define ng Mga Tool gamit ang @tool Decorator

Ang `@tool` decorator ay nagtatakda ng isang simpleng Python function bilang isang tool na maaaring tawagin ng isang ahente.
Pangunahing punto:

- Ang **docstring** ang nagiging deskripsyon ng tool na nakikita ng modelo.
- Ang **Type annotations** (kasama ang `Annotated` na may mga deskripsyon) ay nagtatakda ng schema ng tool.
- Ang `approval_mode` ay kumokontrol kung kinakailangan ng pahintulot ng user bago isagawa ang bawat tawag.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Paglikha ng Agent na may Maramihang Kagamitan

Ibigay ang lahat ng tatlong kagamitan sa kliyente upang ang modelo ay makatawag ng alinman sa mga ito na kailangan upang sagutin ang tanong ng gumagamit.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output with Tools

Sa pamamagitan ng pagtatakda ng `response_format` sa isang Pydantic model, pinipilit ang ahente na magbalik ng maayos na naka-type na JSON object sa halip na malayang anyo ng teksto. Kapaki-pakinabang ito kapag kailangang gamitin ng downstream na code ang resulta sa programmatic na paraan.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mga Pattern ng Pag-apruba ng Tool

Kinokontrol ng parameter na `approval_mode` sa `@tool` kung kailangan ba ng pag-apruba ng tao bago isagawa ang mga tawag sa tool:

| Mode | Pag-uugali |
|---|---|
| `"never_require"` | Awtomatikong tumatakbo ang tool — hindi kailangan ng kumpirmasyon ng gumagamit. |
| `"always_require"` | Bawat tawag ay kailangang aprubahan ng gumagamit bago ito isagawa. |

Gamitin ang `"always_require"` para sa mga tool na may mga side-effect (hal. pag-book ng flight, pagsingil ng credit card) upang manatiling kasama ang tao sa proseso.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

Sa leksyong ito natutunan mo kung paano:

1. **Mag-defina ng mga tools** gamit ang `@tool` decorator na may typed parameters at docstrings na nagsisilbing schema ng tool.
2. **Pagsamahin ang maraming tools** upang tawagan ito ng agent nang sunud-sunod para sagutin ang mga kumplikadong tanong.
3. **Magbalik ng istrukturadong output** sa pamamagitan ng pagpapasa ng modelong Pydantic bilang `response_format`.
4. **Kontrolin ang pag-apruba ng tool** gamit ang `approval_mode` upang mapanatili ang pagkakaroon ng tao sa proseso para sa mga sensitibong operasyon.

Ang mga pattern na ito ang pundasyon para sa paggawa ng maaasahan at handang gamitin sa produksyon na mga agent na kayang makipag-ugnayan sa mga panlabas na sistema nang ligtas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
